# 이안류 CCTV 데이터셋 YOLO 객체 탐지

이 노트북은 이안류 CCTV 데이터셋의 JSON 라벨을 YOLO Detect 형식으로 변환하고, YOLO 모델을 학습·평가·추론하는 과정을 정리한다.

진행 순서는 다음과 같다.

1. 데이터셋과 JSON 구조 확인
2. JSON Bounding Box를 YOLO 형식으로 변환
3. Train / Validation 데이터셋 준비
4. `data.yaml` 작성
5. YOLO 모델 학습
6. Precision, Recall, mAP 확인
7. `best.pt`로 이미지 추론

## 1. 모듈과 경로 설정

`Training`과 `Validation`으로 제공된 원본 분할을 그대로 사용한다. CCTV 연속 프레임을 무작위로 다시 나누면 비슷한 장면이 학습과 검증에 함께 들어가는 데이터 누수가 발생할 수 있다.

In [ ]:
import json
import random
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch
import yaml
from ultralytics import YOLO

PROJECT_ROOT = Path('/Users/songjeong-geun/Desktop/KDT/과제/이안류 CCTV 데이터')
DATA_ROOT = PROJECT_ROOT / '290.이안류 CCTV 데이터' / '01-1.정식개방데이터'

TRAIN_IMAGE_ROOT = DATA_ROOT / 'Training' / '01.원천데이터'
TRAIN_JSON_ROOT = DATA_ROOT / 'Training' / '02.라벨링데이터'
VAL_IMAGE_ROOT = DATA_ROOT / 'Validation' / '01.원천데이터'
VAL_JSON_ROOT = DATA_ROOT / 'Validation' / '02.라벨링데이터'

YOLO_ROOT = PROJECT_ROOT / 'yolo_dataset'
RUN_ROOT = PROJECT_ROOT / 'runs' / 'detect'

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print('DEVICE:', DEVICE)
print('DATA_ROOT:', DATA_ROOT)

## 2. 데이터셋 확인

이미지와 JSON 파일 수를 확인하고, JSON 한 개를 열어 구조를 살펴본다. `annotations.class`가 `1`이고 `object`가 `Yes`이면 이안류가 있는 이미지다. `drawing`에는 이안류 영역을 둘러싼 점 좌표가 들어 있다.

In [ ]:
print(f"Train 이미지: {sum(1 for _ in TRAIN_IMAGE_ROOT.rglob('*.jpg')):,}장")
print(f"Train JSON: {sum(1 for _ in TRAIN_JSON_ROOT.rglob('*.json')):,}개")
print(f"Validation 이미지: {sum(1 for _ in VAL_IMAGE_ROOT.rglob('*.jpg')):,}장")
print(f"Validation JSON: {sum(1 for _ in VAL_JSON_ROOT.rglob('*.json')):,}개")

In [ ]:
sample_json_path = next(TRAIN_JSON_ROOT.rglob('*.json'))

with open(sample_json_path, 'r', encoding='utf-8') as file:
    sample_json = json.load(file)

print(json.dumps(sample_json, ensure_ascii=False, indent=2))

## 3. JSON 라벨을 YOLO Detect 형식으로 변환

YOLO Detect 라벨은 객체 하나를 다음 형식의 한 줄로 표현한다.

```text
class_id x_center y_center width height
```

좌표는 이미지 크기를 기준으로 `0~1` 사이의 값으로 정규화한다. 이안류 클래스는 한 개이므로 클래스 번호는 `0`을 사용한다.

In [ ]:
def box_to_yolo(points, image_width, image_height):
    x_values = [point[0] for point in points]
    y_values = [point[1] for point in points]

    xmin = min(x_values)
    xmax = max(x_values)
    ymin = min(y_values)
    ymax = max(y_values)

    x_center = (xmin + xmax) / 2 / image_width
    y_center = (ymin + ymax) / 2 / image_height
    width = (xmax - xmin) / image_width
    height = (ymax - ymin) / image_height

    return x_center, y_center, width, height

In [ ]:
def prepare_split(image_root, json_root, split_name, positive_limit, background_limit):
    output_image_dir = YOLO_ROOT / 'images' / split_name
    output_label_dir = YOLO_ROOT / 'labels' / split_name
    output_image_dir.mkdir(parents=True, exist_ok=True)
    output_label_dir.mkdir(parents=True, exist_ok=True)

    image_paths = {path.name: path for path in image_root.rglob('*.jpg')}
    json_paths = list(json_root.rglob('*.json'))
    random.shuffle(json_paths)

    positive_count = 0
    background_count = 0

    for json_path in json_paths:
        with open(json_path, 'r', encoding='utf-8') as file:
            data = json.load(file)

        annotation = data['annotations']
        is_positive = annotation['class'] == 1 and annotation['object'] == 'Yes'

        if is_positive and positive_count >= positive_limit:
            continue
        if not is_positive and background_count >= background_limit:
            continue

        file_name = data['image_info']['file_name']
        image_path = image_paths[file_name]
        shutil.copy2(image_path, output_image_dir / file_name)

        width, height = map(int, data['image_info']['resolution'].split(','))
        label_lines = []

        if is_positive:
            for points in annotation['drawing']:
                x, y, w, h = box_to_yolo(points, width, height)
                label_lines.append(f'0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
            positive_count += 1
        else:
            background_count += 1

        label_path = output_label_dir / f'{Path(file_name).stem}.txt'
        label_path.write_text('\n'.join(label_lines), encoding='utf-8')

        if positive_count == positive_limit and background_count == background_limit:
            break

    print(f'{split_name}: 이안류 {positive_count}장, 배경 {background_count}장')

In [ ]:
random.seed(2026)

# 블로그에서 정리한 샘플 규모를 기준으로 구성한다.
prepare_split(TRAIN_IMAGE_ROOT, TRAIN_JSON_ROOT, 'train', 240, 120)
prepare_split(VAL_IMAGE_ROOT, VAL_JSON_ROOT, 'val', 60, 30)

## 4. 변환한 라벨 확인

학습 전에 변환한 Bounding Box가 실제 이안류 위치와 맞는지 확인한다.

In [ ]:
label_path = next(path for path in (YOLO_ROOT / 'labels' / 'train').glob('*.txt') if path.stat().st_size > 0)
image_path = YOLO_ROOT / 'images' / 'train' / f'{label_path.stem}.jpg'

image = cv2.imread(str(image_path))
height, width = image.shape[:2]

for line in label_path.read_text(encoding='utf-8').splitlines():
    _, x, y, w, h = map(float, line.split())
    xmin = int((x - w / 2) * width)
    ymin = int((y - h / 2) * height)
    xmax = int((x + w / 2) * width)
    ymax = int((y + h / 2) * height)
    cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 3)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## 5. `data.yaml` 작성

Dataset YAML에는 데이터셋 경로, 학습·검증 이미지 폴더, 클래스 이름을 기록한다.

In [ ]:
data_yaml = YOLO_ROOT / 'data.yaml'

data_config = {
    'path': str(YOLO_ROOT.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: '이안류'},
}

with open(data_yaml, 'w', encoding='utf-8') as file:
    yaml.safe_dump(data_config, file, allow_unicode=True, sort_keys=False)

print(data_yaml.read_text(encoding='utf-8'))

## 6. YOLO 모델 학습

사전 학습된 `yolo11s.pt`를 불러와 이안류 데이터셋으로 전이 학습한다. 학습이 끝나면 가장 좋은 가중치가 `weights/best.pt`에 저장된다.

In [ ]:
model = YOLO('yolo11s.pt')

model.train(
    data=str(data_yaml),
    epochs=10,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2,
    project=str(RUN_ROOT),
    name='rip_current_yolo11s',
    exist_ok=True,
    seed=2026,
)

## 7. 학습 결과와 평가 지표 확인

- **Loss**: 예측과 정답의 차이다. 학습이 진행되며 줄어드는지 확인한다.
- **Precision**: 모델이 이안류라고 탐지한 것 중 실제 이안류의 비율이다.
- **Recall**: 실제 이안류 중 모델이 찾아낸 비율이다.
- **mAP50**: IoU 0.5를 기준으로 계산한 평균 정밀도다.
- **mAP50-95**: IoU 0.5부터 0.95까지 계산한 더 엄격한 지표다.

In [ ]:
best_path = RUN_ROOT / 'rip_current_yolo11s' / 'weights' / 'best.pt'
best_model = YOLO(str(best_path))

val_results = best_model.val(
    data=str(data_yaml),
    split='val',
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2,
)

print('best.pt:', best_path)
print('Precision:', val_results.box.mp)
print('Recall:', val_results.box.mr)
print('mAP50:', val_results.box.map50)
print('mAP50-95:', val_results.box.map)

In [ ]:
results_image = RUN_ROOT / 'rip_current_yolo11s' / 'results.png'

plt.figure(figsize=(14, 8))
plt.imshow(plt.imread(results_image))
plt.axis('off')
plt.show()

## 8. `best.pt`로 이미지 추론

검증 이미지 한 장을 사용해 추론한다. 결과에는 이안류 Bounding Box와 Confidence Score가 표시된다. 다른 이미지나 영상을 사용하려면 `source` 경로만 바꾸면 된다.

In [ ]:
source = next((YOLO_ROOT / 'images' / 'val').glob('*.jpg'))

pred_results = best_model.predict(
    source=str(source),
    imgsz=640,
    conf=0.25,
    device=DEVICE,
    save=True,
    save_txt=True,
    save_conf=True,
    project=str(RUN_ROOT),
    name='rip_current_predict',
    exist_ok=True,
)

result_image = pred_results[0].plot()

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

### 새로운 이미지 또는 영상 추론

아래 `source`에 새 CCTV 이미지나 영상 경로를 넣어 실행한다.

```python
source = PROJECT_ROOT / '새로운_CCTV_영상.mp4'

best_model.predict(
    source=str(source),
    conf=0.25,
    device=DEVICE,
    save=True,
)
```

저장된 최종 결과에서 Bounding Box와 Confidence Score를 확인할 수 있다. 이안류는 형태와 경계가 불규칙하므로, Detect 결과를 확인한 뒤 더 정확한 영역 표현이 필요하면 Segmentation 방식도 검토할 수 있다.